# YT Agent — Kaggle GPU worker

Triggered by the `kaggle-dispatch.yml` GitHub Actions workflow whenever the
Vercel `/api/maintenance/needs-worker` route reports that a render is queued
and no GPU worker is alive. Boots an ephemeral FastAPI backend on a free
Kaggle T4 / P100, registers itself in Firestore, claims the queued job, runs
the pipeline, then self-terminates after 10 min of idle to preserve the
30 GPU hr/week budget.

**One-time setup**: see `kaggle/README.md`. You need to add
`GOOGLE_APPLICATION_CREDENTIALS_JSON` (and optionally R2 / SFTP) to the
Kaggle Add-ons → Secrets panel for THIS notebook.

In [ ]:
# 1) System deps + repo clone
import os, subprocess
subprocess.check_call(['apt-get', '-qq', 'update'])
subprocess.check_call(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])

REPO_URL = 'https://github.com/Ahsan3301/yt_agent.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/yt_agent'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at:', os.getcwd())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 2) Python deps — base + GPU extras (torch CUDA, decord, av)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
# Kaggle's base image already includes a CUDA torch; install the GPU
# extras NOT pinned to a specific torch wheel (skip torch line to
# avoid clobbering Kaggle's preinstalled CUDA torch).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'decord==0.6.0', 'av==12.3.0'])
import torch
print('torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())

In [ ]:
# 3) Bootstrap secrets via Kaggle Secrets API.
#
# Kaggle's UI detaches secrets from each new notebook version on push,
# which made keeping R2/SFTP credentials attached impossible (every
# kaggle kernels push from GitHub Actions = unattached secrets). We
# fixed that by moving R2 + SFTP into Firestore via the dashboard's
# Connections page. Now the ONLY platform-level secret this notebook
# needs is the Firebase service-account (the chicken-and-egg one).
#
# Add this ONE secret in Kaggle: Notebook → Add-ons → Secrets:
#
#   GOOGLE_APPLICATION_CREDENTIALS_JSON_B64
#       base64-encoded Firebase service-account JSON. Kaggle truncates
#       plain text values >150 chars, so base64 is required here.
#
#   To create on Windows PowerShell:
#     $j = Get-Content path\to\firebase-sa.json -Raw
#     [Convert]::ToBase64String([Text.Encoding]::UTF8.GetBytes($j)) | clip
#
#   On Linux/Mac:
#     base64 -w0 path/to/firebase-sa.json
#
# Everything else (R2_*, SFTP_*, NIM, Groq, Pexels, Discord webhook,
# etc.) comes from Firestore api_keys, populated via the dashboard's
# Connections page → backend/keys_sync.pull_into_env() at startup.
from kaggle_secrets import UserSecretsClient
import os
_sec = UserSecretsClient()

# Try both variants — _B64 is the recommended path on Kaggle.
for k in ('GOOGLE_APPLICATION_CREDENTIALS_JSON_B64',
          'GOOGLE_APPLICATION_CREDENTIALS_JSON'):
    try:
        v = _sec.get_secret(k)
        if v:
            os.environ[k] = str(v)
    except Exception:
        pass

if not (os.environ.get('GOOGLE_APPLICATION_CREDENTIALS_JSON_B64')
        or os.environ.get('GOOGLE_APPLICATION_CREDENTIALS_JSON')):
    raise RuntimeError(
        'Missing Firebase credential. Add GOOGLE_APPLICATION_CREDENTIALS_JSON_B64 '
        'to Notebook → Add-ons → Secrets. This is the only platform-level secret '
        'Kaggle needs — everything else (R2, SFTP, NIM, Groq, etc.) lives in '
        'Firestore and is pulled at server startup.'
    )

# Identity. backend/registry auto-detects the actual GPU model via
# nvidia-smi and appends it, so the dashboard shows e.g.
# "kaggle · Tesla P100-PCIE-16GB" — accurate regardless of which
# GPU Kaggle assigned this session.
os.environ['INSTANCE_TIER']  = 'gpu'
os.environ['INSTANCE_LABEL'] = 'kaggle'
# Self-terminate after 10 min of idle so we don't waste Kaggle GPU hours.
os.environ['KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS'] = '600'

print('Bootstrap: Firebase creds loaded; everything else will be pulled from Firestore at server startup.')

In [ ]:
# 4) Cloudflare quick tunnel — same pattern as Colab cell 6.
import os, subprocess, time, re
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.check_call([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared',
    ])
    subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])
tunnel_log = '/tmp/cloudflared.log'
subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
     '--logfile', tunnel_log, '--loglevel', 'info'],
    stdout=open(tunnel_log + '.stdout', 'w'), stderr=subprocess.STDOUT,
)
url = None
for _ in range(40):
    time.sleep(0.5)
    if not os.path.exists(tunnel_log):
        continue
    with open(tunnel_log) as f:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
        if m:
            url = m.group(0); break
if not url:
    raise RuntimeError('cloudflared did not produce a URL')
os.environ['PUBLIC_BACKEND_URL'] = url
print('Public backend URL:', url)

In [ ]:
# 5) Boot the backend (BLOCKING). idle_watchdog will os._exit(0) after
#    KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS once the queue is empty.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])